# 01 — WorldPop + H3 population exploration

> **Required runtime:** this notebook uses `geolibre_lite.LiteMap`, not upstream `geolibre.Map`.


In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install("geolibre==3.0.0")
from geolibre_lite import LiteMap as Map
assert Map.__module__ == "geolibre_lite", f"Wrong Map loaded: {Map.__module__}"
print("Map implementation:", Map.__module__)


In [ ]:
WORLDPOP_H3 = "https://data.source.coop/smartmaps/h3ys-worldpop/khm.pmtiles"

m = Map(center=(104.92, 12.55), zoom=6, height="700px")
m.add_pmtiles(WORLDPOP_H3, name="WorldPop population — H3 PMTiles")
m


## Create a real H3 cell in the browser

Current Pyodide distributions include the `h3` package, so we can use Uber's H3 indexing without a server.


In [ ]:
import h3

lat, lon = 11.5564, 104.9282  # Phnom Penh
cell = h3.latlng_to_cell(lat, lon, 7)
boundary = h3.cell_to_boundary(cell)

feature = {
    "type": "Feature",
    "properties": {"h3": cell, "resolution": 7, "place": "Phnom Penh"},
    "geometry": {
        "type": "Polygon",
        "coordinates": [[[lng, lat] for lat, lng in boundary] + [[boundary[0][1], boundary[0][0]]]],
    },
}
cell


In [ ]:
m.add_geojson(
    {"type": "FeatureCollection", "features": [feature]},
    name="Example H3 r7 cell",
    fillColor="#ffcc00",
    fillOpacity=0.25,
    strokeColor="#111111",
    strokeWidth=2,
)
m.add_marker(lon, lat, name="Phnom Penh", properties={"h3": cell})
m


## Ideas to extend

- Use `h3.grid_disk(cell, k)` to build neighborhoods.
- Join indicators by H3 index instead of doing repeated polygon overlays.
- Compare population totals at multiple H3 resolutions to demonstrate scale effects.
- Use GeoLibre's PMTiles styling tools to create a graduated population map.

## Data & software citations

- WorldPop 2020 population-count product family: Bondarenko, M., Kerr, D., Sorichetta, A., & Tatem, A.J. (2020), WorldPop, University of Southampton. https://doi.org/10.5258/SOTON/WP00684
- WorldPop-on-H3 PMTiles mirror: UN Smart Maps Group, Source Cooperative, https://source.coop/smartmaps/h3ys-worldpop/khm.pmtiles
- H3: https://h3geo.org/ and https://github.com/uber/h3-py (Apache-2.0).
- GeoLibre: https://geolibre.app/.
